# Expressions - Rust

All 8 Rust examples from [docs/expression.md](https://platob.github.io/yggdryl/expression/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

## The four stages

In [ ]:
use std::str::FromStr;

use yggdryl::{Expression, Field, Scalar};

let schema = Field::from_str("trades:struct<ccy:utf8,price:decimal(9,2),size:bigint>")?;
let filter: Expression = "ccy = 'EUR' and price > 100".parse()?;

// Text is the canonical form, and it re-parses to the same tree.
assert_eq!(filter.to_string(), "ccy = 'EUR' and price > 100");
assert_eq!(filter.columns(), vec!["ccy".to_owned(), "price".to_owned()]);

let bound = filter.bind(&schema)?;
// The literal was converted once, into the column's own exact type.
assert_eq!(
    bound.expression().to_string(),
    "ccy = 'EUR' and price > decimal128(9,2) '100.00'",
);

let row = Scalar::from_sequence([
    Scalar::from("EUR"),
    Scalar::D128(15_000, 2),
    Scalar::I64(5),
]);
assert!(bound.matches(&row)?);

## Text round-trips

In [ ]:
use yggdryl::Expression;

for text in [
    "x is not null",
    "x is distinct from y",
    "x in (1, 2, 3)",
    "x between 1 and 10",
    "name like 'a%' escape '\\'",
    "path glob '**/*.parquet'",
    "case when a then 1 else 2 end",
    "trade.legs[0]['ccy'] = 'EUR'",
    "try_cast(x as decimal128(9,2)) > decimal128(9,2) '1.50'",
] {
    let parsed: Expression = text.parse()?;
    assert_eq!(parsed.to_string().parse::<Expression>()?, parsed);
}

## Null is unknown

In [ ]:
use yggdryl::{Expression, Field, Scalar};

let schema: Field = "rows:struct<a:bigint>".parse()?;
let bound = "a > 1".parse::<Expression>()?.bind(&schema)?;

let missing = Scalar::from_sequence([Scalar::Null]);
assert_eq!(bound.eval(&missing)?, Scalar::Null); // the answer is unknown
assert!(!bound.matches(&missing)?); // and unknown does not keep the row

## `&holder.*`: asking about the file

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::local::Folder;
use yggdryl::Expression;

let lake = Folder::new(std::env::temp_dir().join("yggdryl-docs-lake"))?;
std::fs::create_dir_all(lake.path()?.join("year=2024"))?;
std::fs::write(lake.path()?.join("year=2024").join("part-0.parquet"), b"")?;
std::fs::create_dir_all(lake.path()?.join("year=2025"))?;
std::fs::write(lake.path()?.join("year=2025").join("part-0.parquet"), b"")?;

let filter: Expression = "&holder.partition['year'] = '2024'".parse()?;
let matched: Vec<_> = lake
    .children_matching(&filter, false)?
    .collect::<yggdryl::Result<_>>()?;
assert!(!matched.is_empty());
assert!(matched.iter().all(|entry| {
    entry.url().is_some_and(|url| url.to_string().contains("year=2024"))
}));

std::fs::remove_dir_all(lake.path()?)?;

## Pruning: answering without reading

In [ ]:
use yggdryl::expression::Bounds;
use yggdryl::{Expression, Field, Scalar};

let schema: Field = "trades:struct<ccy:utf8,size:bigint>".parse()?;
let bounds = Bounds::new(Some(1_000))
    .with_column("ccy", Some(Scalar::from("EUR")), Some(Scalar::from("USD")), Some(0))
    .with_column("size", Some(Scalar::I64(1)), Some(Scalar::I64(99)), Some(4));

// Provably empty: no row can hold a size above the file's maximum.
assert!(!"size > 1000".parse::<Expression>()?.bind(&schema)?.statistics_prune(&bounds));
// Not provable either way: the range overlaps, so the file is read.
assert!("size > 50".parse::<Expression>()?.bind(&schema)?.statistics_prune(&bounds));
// A null test the count settles outright.
assert!("size is null".parse::<Expression>()?.bind(&schema)?.statistics_prune(&bounds));

In [ ]:
use yggdryl::{Expression, Field};

let mut schema: Field = "trades:struct<year:int32,price:decimal(9,2)>".parse()?;
let mut children = schema.fields().to_vec();
children[0].set_partition(true);
schema.set_dtype(yggdryl::DataType::from_fields(children)?)?;

let bound = "year = 2024 and price > 100".parse::<Expression>()?.bind(&schema)?;
let residual = bound.partition_split();
assert_eq!(residual.answerable().to_string(), "year = int32 '2024'");
assert_eq!(residual.remaining().to_string(), "price > decimal128(9,2) '100.00'");
assert!(!residual.is_complete());

## Bind a whole statement once

In [ ]:
use yggdryl::expression::Statement;
use yggdryl::Field;

let field: Field = "rows:struct<ccy:utf8,size:bigint>".parse()?;
let statement: Statement = "select ccy, size as quantity where size >= 2 limit 10".parse()?;
let bound = statement.bind(&field)?;
assert_eq!(bound.output().fields()[1].name(), "quantity");

## Targets own their application

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::expression::{ApplyExpression, ApplyExpressionStream, Attributes, Bounds};
use yggdryl::{Expression, Field, Url, Scalar};

// Non-nullable at the root, because the batch below projects it to Arrow.
let schema = "trades:struct<ccy:utf8,size:bigint>".parse::<Field>()?.with_nullable(false);
let bound = "ccy = 'EUR' and size > 10".parse::<Expression>()?.bind(&schema)?;

// One row applies to the value the expression computes.
let row = Scalar::from_sequence([Scalar::from("EUR"), Scalar::I64(25)]);
assert_eq!(row.apply_expression(&bound)?, Scalar::Bool(true));

// One batch applies to one column of answers, one per row.
let arrow_schema = schema.into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(StringArray::from(vec!["EUR", "USD"])),
        Arc::new(Int64Array::from(vec![25_i64, 25])),
    ],
)?;
assert_eq!(batch.apply_expression(&bound)?.len(), 2);

// Statistics apply to the certainty pruning runs on: every size is below 10,
// so no row can match and the container is skipped unread.
let bounds = Bounds::new(Some(1_000))
    .with_column("size", Some(Scalar::I64(1)), Some(Scalar::I64(5)), Some(0));
assert_eq!(bounds.apply_expression(&bound)?, Some(false));

// A holder settles only the conjuncts that need no row - here none - and an
// unknown answer excludes nothing.
let url = Url::from_str("file:///lake/year=2024/part-0.parquet")?;
let holder: &dyn Attributes = &url;
assert_eq!(holder.apply_expression(&bound)?, Scalar::Null);

// The stream sibling consumes its reader, and its application is the
// filtering reader: only the EUR row above survives.
let filtered = yggdryl::arrow::batch_reader(arrow_schema, [batch])
    .apply_expression_stream(&bound)?;
let mut kept = 0;
for batch in filtered {
    kept += batch?.num_rows();
}
assert_eq!(kept, 1);